# LFS Reporting Framework — Demo Runbook

This notebook walks through the complete **Loss Forecast Suite (LFS)** reporting
pipeline using **fully synthetic mock data**. No production tables are read or written.

### What this notebook covers

| Layer | Purpose | Key outputs |
|---|---|---|
| **Layer 1** | Account-level enrichment | Deciles, bands, buckets, DQ flags |
| **Layer 2** | Aggregated business tables | Summaries by channel, source, line, bucket |
| **Layer 3** | Model monitoring | Score drift, feature PSI, data quality, population mix |

### How to connect to real data
Replace the mock-data cells with a call to `runner.run()` (see `docs/notebook_runbook.md`)
once you are ready to run against the production catalog.

## 1. Setup

In [1]:
import os, sys
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ.get("PATH", "")

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("python =", sys.version)

# Resolve project root (one level above this notebooks/ directory).
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

# Fallback: if running from project root directly.
if not os.path.isdir(os.path.join(PROJECT_ROOT, "framework")):
    PROJECT_ROOT = NOTEBOOK_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"sys.path[0]:  {sys.path[0]}")

JAVA_HOME = /opt/homebrew/opt/openjdk@17
python = 3.14.1 (v3.14.1:57e0d177c26, Dec  2 2025, 08:45:31) [Clang 16.0.0 (clang-1600.0.26.6)]
Project root: /Users/hq/dev/claudecode/model_reporting
sys.path[0]:  /Users/hq/dev/claudecode/model_reporting


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# In a DSW environment, `spark` is already available.
# Uncomment the block below only when running locally.
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LFS_Reporting_Demo")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/18 14:46:46 WARN Utils: Your hostname, MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.16.119.45 instead (on interface en0)
26/03/18 14:46:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 14:46:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 ready


In [3]:
from framework.config import load_model_config
from framework.layer1 import enrich_layer1
from framework.layer2 import build_layer2
from framework.layer3 import build_layer3
from notebooks.mock_data import generate_lfs_mock_data

print("Framework imports OK")

Framework imports OK


In [4]:
# Install HTML export and chart dependencies.
# Safe to re-run — pip skips packages that are already installed.
%pip install markdown matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import importlib.util

deps = {
    "markdown"  : importlib.util.find_spec("markdown"),
    "matplotlib": importlib.util.find_spec("matplotlib"),
}
for lib, spec in deps.items():
    status = "installed" if spec is not None else "NOT installed"
    print(f"  {lib:<12} {status}")

print()
html_ok  = deps["markdown"] is not None
chart_ok = deps["matplotlib"] is not None
print(f"HTML export  : {'supported' if html_ok  else 'not supported  →  pip install markdown'}")
print(f"Charts       : {'supported' if chart_ok else 'not supported  →  pip install matplotlib'}")

  markdown     installed
  matplotlib   installed

HTML export  : supported
Charts       : supported


## 2. Synthetic Mock Data

The `mock_data` module generates two DataFrames that match the LFS source schema:

- **`baseline_df`** — 7 months (2024-08 → 2025-02), 2 channels × 250 accounts/month  
  Used for static bin edges and Layer 3 baseline metrics.
- **`current_df`** — 3 months (2025-03 → 2025-05), 2 channels × 200 accounts/month  
  The scored population being reported on.

**Built-in drift signals** so monitoring tables are non-trivial:

| Signal | Description |
|---|---|
| Score mean +0.06–0.08 | Mild upward shift current vs baseline |
| Score std +0.03–0.04 | Slight tail thickening |
| `feature_03` mean +0.14 | Largest feature drift — should flag in feature PSI |
| `feature_01` mean +0.06 | Secondary feature drift |
| `feature_10` mean +0.08 | Moderate feature drift |
| Source mix shift | "organic" −10 pp, "paid"/"referral" each +5 pp |
| Channel difference | digital scores ~0.04–0.06 higher than directmail |

In [6]:
data = generate_lfs_mock_data(
    spark,
    n_baseline_per_month_channel=250,
    n_current_per_month_channel=200,
    seed=42,
)
baseline_df = data["baseline_df"]
current_df  = data["current_df"]

Generated baseline_df: 3,500 rows  (7 months × 2 channels × 250)
Generated current_df:  1,200 rows  (3 months × 2 channels × 200)


In [7]:
print("=== Baseline: score stats by channel ===")
(baseline_df
 .groupBy("channel")
 .agg(
     F.count("*").alias("n_accounts"),
     F.round(F.avg("lfs_score"), 4).alias("mean_score"),
     F.round(F.stddev("lfs_score"), 4).alias("std_score"),
     F.round(F.min("lfs_score"), 4).alias("min_score"),
     F.round(F.max("lfs_score"), 4).alias("max_score"),
 )
 .orderBy("channel")
 .show()
)

print("=== Current: score stats by (month, channel) ===")
(current_df
 .groupBy("month", "channel")
 .agg(
     F.count("*").alias("n_accounts"),
     F.round(F.avg("lfs_score"), 4).alias("mean_score"),
     F.round(F.stddev("lfs_score"), 4).alias("std_score"),
 )
 .orderBy("month", "channel")
 .show()
)

=== Baseline: score stats by channel ===
+----------+----------+----------+---------+---------+---------+
|   channel|n_accounts|mean_score|std_score|min_score|max_score|
+----------+----------+----------+---------+---------+---------+
|   digital|      1750|    0.3831|    0.166|     0.01|     0.99|
|directmail|      1750|    0.3184|   0.1484|     0.01|   0.7616|
+----------+----------+----------+---------+---------+---------+

=== Current: score stats by (month, channel) ===
+-------+----------+----------+----------+---------+
|  month|   channel|n_accounts|mean_score|std_score|
+-------+----------+----------+----------+---------+
|2025-03|   digital|       200|    0.4343|   0.2012|
|2025-03|directmail|       200|    0.4092|   0.2144|
|2025-04|   digital|       200|    0.4628|   0.2098|
|2025-04|directmail|       200|    0.3988|   0.2121|
|2025-05|   digital|       200|    0.4205|   0.2115|
|2025-05|directmail|       200|    0.4035|   0.2081|
+-------+----------+----------+----------+

In [8]:
print("=== Source mix: baseline vs current ===")
base_mix = (baseline_df
    .groupBy("source")
    .agg(F.count("*").alias("n"))
    .withColumn("pct", F.round(F.col("n") / baseline_df.count(), 3))
    .withColumn("period", F.lit("baseline"))
)
curr_mix = (current_df
    .groupBy("source")
    .agg(F.count("*").alias("n"))
    .withColumn("pct", F.round(F.col("n") / current_df.count(), 3))
    .withColumn("period", F.lit("current"))
)
base_mix.union(curr_mix).orderBy("period", "source").show()

=== Source mix: baseline vs current ===
+--------+----+-----+--------+
|  source|   n|  pct|  period|
+--------+----+-----+--------+
| organic|1735|0.496|baseline|
|    paid|1047|0.299|baseline|
|referral| 718|0.205|baseline|
| organic| 480|  0.4| current|
|    paid| 418|0.348| current|
|referral| 302|0.252| current|
+--------+----+-----+--------+



## 3. Load LFS Configuration

The pipeline is entirely config-driven.  `load_model_config("lfs")` reads
`conf/models/lfs.yaml` and returns a typed `ModelConfig` object — no code
changes needed when tuning thresholds, adding features, or renaming tables.

In [9]:
config = load_model_config("lfs")

print(f"Model         : {config.name} — {config.display_name}")
print(f"Channels      : {config.source['channels']}")
print(f"Score column  : {config.source['score_col']}")
print(f"Features      : {config.layer3['feature_columns']}")
print()

l2_tables = (
    list(config.layer2.get("standard_tables", {}).keys())
    + list(config.layer2.get("driver_tables", {}).keys())
)
print(f"Layer 2 tables ({len(l2_tables)}):")
for t in l2_tables:
    print(f"  {config.layer2.get('standard_tables', {}).get(t, config.layer2.get('driver_tables', {}).get(t, {})).get('output_table', t)}")

print()
print(f"Layer 3 tables ({len(config.layer3['tables'])}):")
for t, cfg in config.layer3["tables"].items():
    print(f"  {cfg['output_table']}")

Model         : lfs — Loss Forecast Suite
Channels      : ['digital', 'directmail']
Score column  : lfs_score
Features      : ['feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11']

Layer 2 tables (8):
  lfs_layer2_overall_summary
  lfs_layer2_score_distribution
  lfs_layer2_by_source
  lfs_layer2_by_line
  lfs_layer2_by_receivable_bucket
  lfs_layer2_by_saleamount_bucket
  lfs_layer2_by_salecount_bucket
  lfs_layer2_by_mob_pti_bucket

Layer 3 tables (8):
  lfs_layer3_score_drift
  lfs_layer3_feature_psi
  lfs_layer3_data_quality
  lfs_layer3_feature_quality
  lfs_layer3_feature_score_relationship
  lfs_layer3_population_mix
  lfs_layer3_performance
  lfs_layer3_calibration


## 4. Layer 1 — Account-Level Enrichment

`enrich_layer1` adds these columns to every account row:

| Column | Description |
|---|---|
| `score_month` | Pipeline run label (passed in as argument) |
| `model_version` | Version tag |
| `vintage_month` | Per-row copy of the scoring month |
| `lfs_decile_dyn` | Dynamic decile (1–10) computed within each (month, channel) |
| `lfs_band_dyn` | Decile band label: Low / Medium / High |
| `lfs_decile_static` | Static decile using edges from the baseline window |
| `receivable_bucket` | Fixed-boundary bucket for `endingreceivable` |
| `saleamount_bucket` | Fixed-boundary bucket for `saleamount` |
| `salecount_bucket` | Fixed-boundary bucket for `salecount` |
| `mob_pti_bucket` | Fixed-boundary bucket for `mob_pti` |
| `dq_missing_flag` | 1 if `lfs_score` is NULL |
| `dq_outlier_flag` | 1 if `lfs_score` outside [0, 1] |

The current data spans 3 months; Layer 1 enriches all of them in one pass.

In [10]:
SCORE_MONTH   = "2025-05"   # pipeline-run label (latest current month)
MODEL_VERSION = "v1.0"

enriched_df = enrich_layer1(
    current_df,
    config,
    baseline_df,
    score_month=SCORE_MONTH,
    model_version=MODEL_VERSION,
)
print(f"Layer 1 complete — {enriched_df.count():,} rows, {len(enriched_df.columns)} columns")
print(f"New columns added by Layer 1:")
orig_cols = set(current_df.columns)
new_cols  = [c for c in enriched_df.columns if c not in orig_cols]
print("  " + ", ".join(new_cols))

2026-03-18 14:46:51,666 [framework.layer1] INFO: Adding metadata columns
INFO:framework.layer1:Adding metadata columns
2026-03-18 14:46:51,674 [framework.layer1] INFO: Computing dynamic deciles: 10 bins by ['month', 'channel']
INFO:framework.layer1:Computing dynamic deciles: 10 bins by ['month', 'channel']
2026-03-18 14:46:51,693 [framework.layer1] INFO: Mapping deciles to bands
INFO:framework.layer1:Mapping deciles to bands
2026-03-18 14:46:51,716 [framework.layer1] INFO: Computing static deciles from baseline ['2024-08', '2025-02']
INFO:framework.layer1:Computing static deciles from baseline ['2024-08', '2025-02']
2026-03-18 14:46:52,215 [framework.layer1] INFO: Adding bucket: receivable_bucket
INFO:framework.layer1:Adding bucket: receivable_bucket
2026-03-18 14:46:52,254 [framework.layer1] INFO: Adding bucket: saleamount_bucket
INFO:framework.layer1:Adding bucket: saleamount_bucket
2026-03-18 14:46:52,278 [framework.layer1] INFO: Adding bucket: salecount_bucket
INFO:framework.layer1

Layer 1 complete — 1,200 rows, 35 columns
New columns added by Layer 1:
  score_month, model_version, vintage_month, lfs_decile_dyn, lfs_band_dyn, lfs_decile_static, receivable_bucket, saleamount_bucket, salecount_bucket, mob_pti_bucket, dq_missing_flag, dq_outlier_flag


In [11]:
print("=== Layer 1 sample — enriched columns ===")
enriched_df.select(
    "creditaccountid", "vintage_month", "channel",
    "lfs_score",
    "lfs_decile_dyn", "lfs_band_dyn", "lfs_decile_static",
    "receivable_bucket", "saleamount_bucket",
    "dq_missing_flag", "dq_outlier_flag",
).show(15, truncate=False)

=== Layer 1 sample — enriched columns ===
+---------------+-------------+-------+---------+--------------+------------+-----------------+-----------------+-----------------+---------------+---------------+
|creditaccountid|vintage_month|channel|lfs_score|lfs_decile_dyn|lfs_band_dyn|lfs_decile_static|receivable_bucket|saleamount_bucket|dq_missing_flag|dq_outlier_flag|
+---------------+-------------+-------+---------+--------------+------------+-----------------+-----------------+-----------------+---------------+---------------+
|3559           |2025-03      |digital|0.01     |1             |Low         |1                |1K-2.5K          |5K+              |0              |0              |
|3627           |2025-03      |digital|0.01     |1             |Low         |1                |5K-10K           |5K+              |0              |0              |
|3658           |2025-03      |digital|0.01     |1             |Low         |1                |10K+             |5K+              |0      

In [12]:
from pyspark.sql import Window
print("=== Dynamic decile distribution per (vintage_month, channel) ===")
print("Each decile should contain ~10% of accounts within its partition.\n")
(enriched_df
 .groupBy("vintage_month", "channel", "lfs_decile_dyn")
 .agg(F.count("*").alias("n_accounts"))
 .withColumn(
     "pct",
     F.round(
         F.col("n_accounts") /
         F.sum("n_accounts").over(
             Window
             .partitionBy("vintage_month", "channel")
         ),
         3,
     )
 )
 .orderBy("vintage_month", "channel", "lfs_decile_dyn")
 .show(60)
)

=== Dynamic decile distribution per (vintage_month, channel) ===
Each decile should contain ~10% of accounts within its partition.

+-------------+----------+--------------+----------+---+
|vintage_month|   channel|lfs_decile_dyn|n_accounts|pct|
+-------------+----------+--------------+----------+---+
|      2025-03|   digital|             1|        20|0.1|
|      2025-03|   digital|             2|        20|0.1|
|      2025-03|   digital|             3|        20|0.1|
|      2025-03|   digital|             4|        20|0.1|
|      2025-03|   digital|             5|        20|0.1|
|      2025-03|   digital|             6|        20|0.1|
|      2025-03|   digital|             7|        20|0.1|
|      2025-03|   digital|             8|        20|0.1|
|      2025-03|   digital|             9|        20|0.1|
|      2025-03|   digital|            10|        20|0.1|
|      2025-03|directmail|             1|        20|0.1|
|      2025-03|directmail|             2|        20|0.1|
|      2025-0

In [13]:
from pyspark.sql import Window
print("=== Band distribution per channel ===")
(enriched_df
 .groupBy("channel", "lfs_band_dyn")
 .agg(F.count("*").alias("n_accounts"))
 .withColumn(
     "pct_of_channel",
     F.round(
         F.col("n_accounts") /
         F.sum("n_accounts").over(
             Window
             .partitionBy("channel")
         ),
         3,
     )
 )
 .orderBy("channel", "lfs_band_dyn")
 .show()
)

=== Band distribution per channel ===
+----------+------------+----------+--------------+
|   channel|lfs_band_dyn|n_accounts|pct_of_channel|
+----------+------------+----------+--------------+
|   digital|        High|       180|           0.3|
|   digital|         Low|       180|           0.3|
|   digital|      Medium|       240|           0.4|
|directmail|        High|       180|           0.3|
|directmail|         Low|       180|           0.3|
|directmail|      Medium|       240|           0.4|
+----------+------------+----------+--------------+



## 5. Layer 2 — Aggregated Business Tables

`build_layer2` produces 8 tables from the enriched Layer 1 output.  All tables
keep `channel` as a column — no physical split.

**Standard tables** (4):

| Table | Group by |
|---|---|
| `lfs_layer2_overall_summary` | vintage_month, channel |
| `lfs_layer2_score_distribution` | vintage_month, channel, lfs_decile_dyn |
| `lfs_layer2_by_source` | vintage_month, channel, source |
| `lfs_layer2_by_line` | vintage_month, channel, line |

**Driver / drill-down tables** (4):

| Table | Group by |
|---|---|
| `lfs_layer2_by_receivable_bucket` | vintage_month, channel, receivable_bucket |
| `lfs_layer2_by_saleamount_bucket` | vintage_month, channel, saleamount_bucket |
| `lfs_layer2_by_salecount_bucket` | vintage_month, channel, salecount_bucket |
| `lfs_layer2_by_mob_pti_bucket` | vintage_month, channel, mob_pti_bucket |

In [14]:
l2_tables = build_layer2(enriched_df, config)
print(f"Layer 2 complete — {len(l2_tables)} tables produced:")
for tname, tdf in l2_tables.items():
    print(f"  {tname:<45}  {tdf.count():>4} rows  {len(tdf.columns)} cols")

2026-03-18 14:46:53,209 [framework.layer2] INFO: Built lfs_layer2_overall_summary (overall_summary, 11 columns)
INFO:framework.layer2:Built lfs_layer2_overall_summary (overall_summary, 11 columns)
2026-03-18 14:46:53,227 [framework.layer2] INFO: Built lfs_layer2_score_distribution (score_distribution, 7 columns)
INFO:framework.layer2:Built lfs_layer2_score_distribution (score_distribution, 7 columns)
2026-03-18 14:46:53,243 [framework.layer2] INFO: Built lfs_layer2_by_source (by_source, 12 columns)
INFO:framework.layer2:Built lfs_layer2_by_source (by_source, 12 columns)
2026-03-18 14:46:53,259 [framework.layer2] INFO: Built lfs_layer2_by_line (by_line, 12 columns)
INFO:framework.layer2:Built lfs_layer2_by_line (by_line, 12 columns)
2026-03-18 14:46:53,276 [framework.layer2] INFO: Built lfs_layer2_by_receivable_bucket (by_receivable_bucket, 12 columns)
INFO:framework.layer2:Built lfs_layer2_by_receivable_bucket (by_receivable_bucket, 12 columns)
2026-03-18 14:46:53,288 [framework.layer2

Layer 2 complete — 8 tables produced:
  lfs_layer2_overall_summary                        6 rows  11 cols
  lfs_layer2_score_distribution                    60 rows  7 cols
  lfs_layer2_by_source                             18 rows  12 cols
  lfs_layer2_by_line                               12 rows  12 cols
  lfs_layer2_by_receivable_bucket                  36 rows  12 cols
  lfs_layer2_by_saleamount_bucket                  30 rows  12 cols
  lfs_layer2_by_salecount_bucket                   36 rows  12 cols
  lfs_layer2_by_mob_pti_bucket                     36 rows  12 cols


In [15]:
print("=== Overall Summary (lfs_layer2_overall_summary) ===")
print("One row per (vintage_month, channel).  Metrics: count, avg/sum score, receivable, sale.\n")
(l2_tables["lfs_layer2_overall_summary"]
 .select(
     "vintage_month", "channel",
     "account_count",
     F.round("avg_lfs_score", 4).alias("avg_lfs_score"),
     F.round("avg_endingreceivable", 2).alias("avg_recv"),
     F.round("avg_saleamount", 2).alias("avg_saleamt"),
 )
 .orderBy("vintage_month", "channel")
 .show(truncate=False)
)

=== Overall Summary (lfs_layer2_overall_summary) ===
One row per (vintage_month, channel).  Metrics: count, avg/sum score, receivable, sale.

+-------------+----------+-------------+-------------+--------+-----------+
|vintage_month|channel   |account_count|avg_lfs_score|avg_recv|avg_saleamt|
+-------------+----------+-------------+-------------+--------+-----------+
|2025-03      |digital   |200          |0.4343       |7479.65 |2799.58    |
|2025-03      |directmail|200          |0.4092       |6044.3  |2349.5     |
|2025-04      |digital   |200          |0.4628       |6800.84 |2544.43    |
|2025-04      |directmail|200          |0.3988       |6050.71 |2605.96    |
|2025-05      |digital   |200          |0.4205       |7283.25 |2844.92    |
|2025-05      |directmail|200          |0.4035       |5841.38 |2445.5     |
+-------------+----------+-------------+-------------+--------+-----------+



In [16]:
print("=== Score Distribution (lfs_layer2_score_distribution) ===")
print("One row per (vintage_month, channel, decile).  pct_of_channel_vintage_accounts should ~= 0.10.\n")
(l2_tables["lfs_layer2_score_distribution"]
 .select(
     "vintage_month", "channel", "lfs_decile_dyn",
     "account_count",
     F.round("avg_lfs_score", 4).alias("avg_score"),
     F.round("pct_of_channel_vintage_accounts", 4).alias("pct"),
 )
 .orderBy("vintage_month", "channel", "lfs_decile_dyn")
 .show(60, truncate=False)
)

=== Score Distribution (lfs_layer2_score_distribution) ===
One row per (vintage_month, channel, decile).  pct_of_channel_vintage_accounts should ~= 0.10.

+-------------+----------+--------------+-------------+---------+---+
|vintage_month|channel   |lfs_decile_dyn|account_count|avg_score|pct|
+-------------+----------+--------------+-------------+---------+---+
|2025-03      |digital   |1             |20           |0.0781   |0.1|
|2025-03      |digital   |2             |20           |0.2189   |0.1|
|2025-03      |digital   |3             |20           |0.2835   |0.1|
|2025-03      |digital   |4             |20           |0.3424   |0.1|
|2025-03      |digital   |5             |20           |0.4178   |0.1|
|2025-03      |digital   |6             |20           |0.4781   |0.1|
|2025-03      |digital   |7             |20           |0.5222   |0.1|
|2025-03      |digital   |8             |20           |0.5923   |0.1|
|2025-03      |digital   |9             |20           |0.6516   |0.1|
|2025

In [17]:
print("=== By Source (lfs_layer2_by_source) ===")
print("Shows how average score and financials differ across acquisition sources.\n")
(l2_tables["lfs_layer2_by_source"]
 .select(
     "vintage_month", "channel", "source",
     "account_count",
     F.round("avg_lfs_score", 4).alias("avg_score"),
     F.round("avg_saleamount", 2).alias("avg_saleamt"),
 )
 .orderBy("vintage_month", "channel", "source")
 .show(truncate=False)
)

=== By Source (lfs_layer2_by_source) ===
Shows how average score and financials differ across acquisition sources.

+-------------+----------+--------+-------------+---------+-----------+
|vintage_month|channel   |source  |account_count|avg_score|avg_saleamt|
+-------------+----------+--------+-------------+---------+-----------+
|2025-03      |digital   |organic |101          |0.4317   |2842.5     |
|2025-03      |digital   |paid    |57           |0.4215   |2545.01    |
|2025-03      |digital   |referral|42           |0.4577   |3041.87    |
|2025-03      |directmail|organic |74           |0.3999   |2390.41    |
|2025-03      |directmail|paid    |78           |0.4332   |2441.86    |
|2025-03      |directmail|referral|48           |0.3843   |2136.33    |
|2025-04      |digital   |organic |80           |0.4594   |2458.79    |
|2025-04      |digital   |paid    |79           |0.4619   |2596.92    |
|2025-04      |digital   |referral|41           |0.4712   |2610.37    |
|2025-04      |direc

## 6. Layer 3 — Model Monitoring

`build_layer3` compares the current period against the baseline window.
Six monitoring tables are produced:

| Table | Rows per vintage | Key metrics |
|---|---|---|
| `lfs_layer3_score_drift` | 1 per (vintage_month, channel) | mean, std, PSI, KS, quantile cutoffs, baseline exceedance % |
| `lfs_layer3_feature_psi` | 11 per (vintage_month, channel) | PSI, mean/std shift per feature |
| `lfs_layer3_data_quality` | 1 per (vintage_month, channel) | Score missing rate, outlier rate |
| `lfs_layer3_feature_quality` | 11 per (vintage_month, channel) | Feature missing rate, outlier rate (baseline-derived bounds) |
| `lfs_layer3_feature_score_relationship` | 11 per (vintage_month, channel) | Pearson correlation: baseline, current, change |
| `lfs_layer3_population_mix` | N segments per (vintage_month, channel) | Segment %, monitors mix shift |

**Expected monitoring signals with this mock data:**
- `score_drift`: PSI ~ 0.05–0.15 (mild drift), KS > 0.05
- `feature_psi`: `feature_03` should show the highest PSI (~0.10–0.20)
- `population_mix`: source "organic" proportion should drop vs baseline

In [18]:
l3_tables = build_layer3(enriched_df, baseline_df, config)
print(f"Layer 3 complete — {len(l3_tables)} tables produced:")
for tname, tdf in l3_tables.items():
    print(f"  {tname:<50}  {tdf.count():>3} rows  {len(tdf.columns)} cols")

2026-03-18 14:46:54,536 [framework.layer3] INFO: Building lfs_layer3_score_drift
INFO:framework.layer3:Building lfs_layer3_score_drift
26/03/18 14:46:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:46:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:46:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:46:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:46:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:46:56 WARN WindowExec: No Pa

Layer 3 complete — 6 tables produced:
  lfs_layer3_score_drift                                6 rows  14 cols
  lfs_layer3_feature_psi                               66 rows  8 cols
  lfs_layer3_data_quality                               6 rows  4 cols
  lfs_layer3_feature_quality                           66 rows  7 cols
  lfs_layer3_feature_score_relationship                66 rows  6 cols
  lfs_layer3_population_mix                            96 rows  6 cols


### 6.6 Performance Monitoring & Calibration

Synthetic actuals are generated from the LFS score using a sigmoid function:
`P(bad | score) = sigmoid(score × 5 − 2.5)`

Higher scores → higher predicted bad probability — consistent with the LFS model's intent.
The actuals are then joined back into Layer 3 via `build_layer3(..., actuals_df=actuals_df)`.

In [19]:
from notebooks.mock_data import generate_lfs_mock_actuals

# Generate synthetic actuals aligned with enriched_df.
actuals_df = generate_lfs_mock_actuals(spark, enriched_df, seed=42)

print("=== Actuals sample ===")
actuals_df.show(5)

# Re-run Layer 3 with actuals to produce performance + calibration tables.
l3_tables_with_perf = build_layer3(enriched_df, baseline_df, config, actuals_df=actuals_df)
print(f"\nLayer 3 with actuals — {len(l3_tables_with_perf)} tables produced:")
for tname, tdf in l3_tables_with_perf.items():
    print(f"  {tname:<55}  {tdf.count():>3} rows  {len(tdf.columns)} cols")

Generated actuals_df: 1,200 rows  (bad rate: 43.6%)
=== Actuals sample ===


2026-03-18 14:47:54,102 [framework.layer3] INFO: Building lfs_layer3_score_drift
INFO:framework.layer3:Building lfs_layer3_score_drift


+---------------+-------------+------+-----+-----+-----+
|creditaccountid|vintage_month|is_bad|edr30|edr60|edr90|
+---------------+-------------+------+-----+-----+-----+
|           3501|      2025-03|     0|    0|    0|    0|
|           3502|      2025-03|     0|    1|    0|    0|
|           3503|      2025-03|     0|    1|    1|    0|
|           3504|      2025-03|     1|    1|    0|    1|
|           3505|      2025-03|     0|    0|    0|    0|
+---------------+-------------+------+-----+-----+-----+
only showing top 5 rows


26/03/18 14:47:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:47:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:47:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:47:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:47:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 14:47:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 1


Layer 3 with actuals — 8 tables produced:
  lfs_layer3_score_drift                                     6 rows  14 cols
  lfs_layer3_feature_psi                                    66 rows  8 cols
  lfs_layer3_data_quality                                    6 rows  4 cols
  lfs_layer3_feature_quality                                66 rows  7 cols
  lfs_layer3_feature_score_relationship                     66 rows  6 cols
  lfs_layer3_population_mix                                 96 rows  6 cols
  lfs_layer3_performance                                     6 rows  10 cols
  lfs_layer3_calibration                                    60 rows  7 cols


In [20]:
print("=== Performance Monitoring (lfs_layer3_performance) ===\n")
(l3_tables_with_perf["lfs_layer3_performance"]
 .select(
     "vintage_month", "channel", "account_count",
     F.round("avg_lfs_score", 4).alias("avg_score"),
     F.round("predicted_bad_rate", 4).alias("pred_bad_rate"),
     F.round("actual_bad_rate", 4).alias("act_bad_rate"),
     F.round("calibration_gap", 4).alias("calib_gap"),
     F.round("edr90", 4).alias("edr90"),
 )
 .orderBy("vintage_month", "channel")
 .show(truncate=False)
)

=== Performance Monitoring (lfs_layer3_performance) ===

+-------------+----------+-------------+---------+-------------+------------+---------+-----+
|vintage_month|channel   |account_count|avg_score|pred_bad_rate|act_bad_rate|calib_gap|edr90|
+-------------+----------+-------------+---------+-------------+------------+---------+-----+
|2025-03      |digital   |200          |0.4343   |0.4343       |0.41        |-0.0243  |0.41 |
|2025-03      |directmail|200          |0.4092   |0.4092       |0.39        |-0.0192  |0.39 |
|2025-04      |digital   |200          |0.4628   |0.4628       |0.475       |0.0122   |0.475|
|2025-04      |directmail|200          |0.3988   |0.3988       |0.4         |0.0012   |0.4  |
|2025-05      |digital   |200          |0.4205   |0.4205       |0.49        |0.0695   |0.49 |
|2025-05      |directmail|200          |0.4035   |0.4035       |0.45        |0.0465   |0.45 |
+-------------+----------+-------------+---------+-------------+------------+---------+-----+



In [21]:
print("=== Calibration (lfs_layer3_calibration) — 2025-05, digital ===\n")
(l3_tables_with_perf["lfs_layer3_calibration"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("channel") == "digital")
 )
 .select(
     "score_bin", "account_count",
     F.round("predicted_rate", 4).alias("pred_rate"),
     F.round("actual_rate", 4).alias("act_rate"),
     F.round("calibration_gap", 4).alias("gap"),
 )
 .orderBy("score_bin")
 .show(truncate=False)
)

=== Calibration (lfs_layer3_calibration) — 2025-05, digital ===

+---------+-------------+---------+--------+-------+
|score_bin|account_count|pred_rate|act_rate|gap    |
+---------+-------------+---------+--------+-------+
|1        |19           |0.0758   |0.1053  |0.0295 |
|2        |18           |0.201    |0.2222  |0.0212 |
|3        |19           |0.2579   |0.3158  |0.0579 |
|4        |19           |0.3149   |0.3158  |9.0E-4 |
|5        |19           |0.3566   |0.2632  |-0.0934|
|6        |14           |0.4008   |0.5714  |0.1706 |
|7        |16           |0.4339   |0.5     |0.0661 |
|8        |14           |0.4906   |0.7857  |0.2951 |
|9        |16           |0.5523   |0.5625  |0.0102 |
|10       |46           |0.72     |0.8478  |0.1278 |
+---------+-------------+---------+--------+-------+



### 6.1 Score Drift

One row per `(vintage_month, channel)`.  Key columns to watch:

- **`score_psi`** — PSI vs baseline. Flag if > 0.10 (yellow) or > 0.25 (red).
- **`score_ks`** — KS statistic. Larger = larger distributional shift.
- **`mean_lfs_score`** — Average score in current period.
- **`pct_accounts_above_p90_baseline`** — % of current accounts above the baseline p90 threshold.
  Rising value → score distribution shifting right.

In [22]:
print("=== Score Drift (lfs_layer3_score_drift) ===\n")
(l3_tables["lfs_layer3_score_drift"]
 .select(
     "vintage_month", "channel",
     F.round("mean_lfs_score", 4).alias("mean_score"),
     F.round("std_lfs_score", 4).alias("std_score"),
     F.round("score_psi", 5).alias("psi"),
     F.round("score_ks", 5).alias("ks"),
     F.round("q50_score", 4).alias("q50"),
     F.round("q90_score", 4).alias("q90"),
     F.round("q95_score", 4).alias("q95"),
     F.round("mean_top10pct_score", 4).alias("top10_mean"),
     F.round("pct_accounts_above_p90_baseline", 4).alias("pct_above_p90_base"),
 )
 .orderBy("vintage_month", "channel")
 .show(truncate=False)
)

=== Score Drift (lfs_layer3_score_drift) ===

+-------------+----------+----------+---------+-------+-------+------+------+------+----------+------------------+
|vintage_month|channel   |mean_score|std_score|psi    |ks     |q50   |q90   |q95   |top10_mean|pct_above_p90_base|
+-------------+----------+----------+---------+-------+-------+------+------+------+----------+------------------+
|2025-03      |digital   |0.4343    |0.2012   |0.2494 |0.19   |0.4491|0.6803|0.7216|0.7481    |0.265             |
|2025-03      |directmail|0.4092    |0.2144   |0.33673|0.23414|0.3958|0.677 |0.7299|0.7854    |0.305             |
|2025-04      |digital   |0.4628    |0.2098   |0.32438|0.234  |0.4691|0.7056|0.7919|0.8134    |0.285             |
|2025-04      |directmail|0.3988    |0.2121   |0.32518|0.22914|0.3759|0.6761|0.7322|0.7571    |0.315             |
|2025-05      |digital   |0.4205    |0.2115   |0.1201 |0.12743|0.3887|0.6949|0.787 |0.7949    |0.23              |
|2025-05      |directmail|0.4035  

### 6.2 Feature PSI

One row per `(vintage_month, channel, feature_name)`.

- **`psi`** — Population Stability Index for this feature. `feature_03` should
  show the highest value given the +0.14 mean drift baked into the mock data.
- **`baseline_mean` / `current_mean`** — Direct comparison of mean values.
- **`baseline_std` / `current_std`** — Spread comparison.

In [23]:
print("=== Feature PSI — sorted by PSI descending (2025-05, digital) ===\n")
(l3_tables["lfs_layer3_feature_psi"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("channel") == "digital")
 )
 .select(
     "feature_name",
     F.round("psi", 5).alias("psi"),
     F.round("baseline_mean", 4).alias("base_mean"),
     F.round("current_mean", 4).alias("curr_mean"),
     (F.round("current_mean", 4) - F.round("baseline_mean", 4)).alias("mean_shift"),
     F.round("baseline_std", 4).alias("base_std"),
     F.round("current_std", 4).alias("curr_std"),
 )
 .orderBy(F.col("psi").desc())
 .show(truncate=False)
)

=== Feature PSI — sorted by PSI descending (2025-05, digital) ===

+------------+-------+---------+---------+----------------------+--------+--------+
|feature_name|psi    |base_mean|curr_mean|mean_shift            |base_std|curr_std|
+------------+-------+---------+---------+----------------------+--------+--------+
|feature_03  |0.72282|0.2983   |0.4273   |0.129                 |0.1487  |0.1511  |
|feature_10  |0.71174|0.4965   |0.5595   |0.063                 |0.2329  |0.2242  |
|feature_01  |0.44053|0.3804   |0.4592   |0.07879999999999998   |0.1115  |0.1343  |
|feature_04  |0.20275|0.4195   |0.4746   |0.05510000000000004   |0.125   |0.1423  |
|feature_02  |0.10843|0.465    |0.475    |0.009999999999999953  |0.1503  |0.168   |
|feature_09  |0.06666|0.4978   |0.4858   |-0.01200000000000001  |0.2315  |0.2286  |
|feature_05  |0.0576 |0.4778   |0.5031   |0.02529999999999999   |0.1787  |0.1928  |
|feature_06  |0.04901|0.5472   |0.5518   |0.0045999999999999375 |0.1936  |0.2152  |
|feature_

### 6.3 Data Quality

One row per `(vintage_month, channel)`.

- **`missing_score_rate`** — Fraction of accounts with NULL score.  Should be 0.0 with clean mock data.
- **`outlier_score_rate`** — Fraction with score outside [0, 1].  Should be 0.0 since mock scores are clamped.

In [24]:
print("=== Data Quality (lfs_layer3_data_quality) ===\n")
(l3_tables["lfs_layer3_data_quality"]
 .select("vintage_month", "channel", "missing_score_rate", "outlier_score_rate")
 .orderBy("vintage_month", "channel")
 .show(truncate=False)
)

=== Data Quality (lfs_layer3_data_quality) ===

+-------------+----------+------------------+------------------+
|vintage_month|channel   |missing_score_rate|outlier_score_rate|
+-------------+----------+------------------+------------------+
|2025-03      |digital   |0.0               |0.0               |
|2025-03      |directmail|0.0               |0.0               |
|2025-04      |digital   |0.0               |0.0               |
|2025-04      |directmail|0.0               |0.0               |
|2025-05      |digital   |0.0               |0.0               |
|2025-05      |directmail|0.0               |0.0               |
+-------------+----------+------------------+------------------+



### 6.4 Feature-Score Relationship Drift

One row per `(vintage_month, channel, feature_name)`.

- **`baseline_corr`** — Pearson r with score in the baseline window.
- **`current_corr`** — Pearson r with score in the current period.
- **`corr_change`** — Signed difference.  Large negative = feature became less predictive.
- `None` values are expected for features with zero variance or fewer than 2 rows in a slice.

In [25]:
print("=== Feature–Score Correlation Drift (2025-05, digital) ===\n")
(l3_tables["lfs_layer3_feature_score_relationship"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("channel") == "digital")
 )
 .select(
     "feature_name",
     F.round("baseline_corr", 4).alias("base_corr"),
     F.round("current_corr", 4).alias("curr_corr"),
     F.round("corr_change", 4).alias("corr_chg"),
 )
 .orderBy(F.abs("corr_change").desc())
 .show(truncate=False)
)

=== Feature–Score Correlation Drift (2025-05, digital) ===

+------------+---------+---------+--------+
|feature_name|base_corr|curr_corr|corr_chg|
+------------+---------+---------+--------+
|feature_04  |0.4939   |0.6841   |0.1902  |
|feature_11  |0.025    |-0.1362  |-0.1612 |
|feature_08  |-0.0308  |-0.1384  |-0.1076 |
|feature_03  |-0.0172  |0.0523   |0.0696  |
|feature_01  |0.8123   |0.8785   |0.0662  |
|feature_10  |0.0381   |-0.0247  |-0.0627 |
|feature_09  |-0.0377  |0.0233   |0.0611  |
|feature_07  |-0.0347  |0.0253   |0.0601  |
|feature_06  |-0.0319  |0.024    |0.0559  |
|feature_05  |0.1915   |0.2461   |0.0545  |
|feature_02  |0.3365   |0.379    |0.0426  |
+------------+---------+---------+--------+



### 6.5 Population Mix

One row per `(vintage_month, channel, segment_type, segment_value)`.
Tracks composition shifts across four segment dimensions: source, line,
receivable bucket, saleamount bucket.

Watch for shifts in `pct_of_channel_accounts` vs the overall baseline mix
(run the same query on `baseline_df` to compare).

In [26]:
print("=== Population Mix — source segment (2025-05) ===\n")
(l3_tables["lfs_layer3_population_mix"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("segment_type") == "source")
 )
 .select(
     "channel", "segment_type", "segment_value",
     "account_count",
     F.round("pct_of_channel_accounts", 4).alias("pct"),
 )
 .orderBy("channel", "segment_value")
 .show(truncate=False)
)

print("=== Population Mix — receivable_bucket segment (2025-05) ===\n")
(l3_tables["lfs_layer3_population_mix"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("segment_type") == "receivable_bucket")
 )
 .select(
     "channel", "segment_value",
     "account_count",
     F.round("pct_of_channel_accounts", 4).alias("pct"),
 )
 .orderBy("channel", "segment_value")
 .show(truncate=False)
)

=== Population Mix — source segment (2025-05) ===

+----------+------------+-------------+-------------+-----+
|channel   |segment_type|segment_value|account_count|pct  |
+----------+------------+-------------+-------------+-----+
|digital   |source      |organic      |83           |0.415|
|digital   |source      |paid         |69           |0.345|
|digital   |source      |referral     |48           |0.24 |
|directmail|source      |organic      |74           |0.37 |
|directmail|source      |paid         |67           |0.335|
|directmail|source      |referral     |59           |0.295|
+----------+------------+-------------+-------------+-----+

=== Population Mix — receivable_bucket segment (2025-05) ===

+----------+-------------+-------------+-----+
|channel   |segment_value|account_count|pct  |
+----------+-------------+-------------+-----+
|digital   |10K+         |65           |0.325|
|digital   |1K-2.5K      |8            |0.04 |
|digital   |2.5K-5K      |29           |0.145|
|dig

## 7. Quick Validation Checks

These inline assertions replicate the checks from `docs/validation_checklist.md`.
Run this cell to confirm all outputs are internally consistent before promoting
to production.

In [27]:
from pyspark.sql import Window

print("Running validation checks...\n")
errors = []

# 1. Layer 1 row count matches current_df.
l1_count = enriched_df.count()
src_count = current_df.count()
if l1_count != src_count:
    errors.append(f"[FAIL] Layer1 row count {l1_count} != source {src_count}")
else:
    print(f"[OK]   Layer1 row count matches source: {l1_count:,}")

# 2. Layer2 account_count sums to Layer1 total.
l2_total = (
    l2_tables["lfs_layer2_overall_summary"]
    .agg(F.sum("account_count").alias("total"))
    .collect()[0]["total"]
)
if l2_total != l1_count:
    errors.append(f"[FAIL] Layer2 account_count sum {l2_total} != Layer1 {l1_count}")
else:
    print(f"[OK]   Layer2 account_count sums to Layer1 total: {l2_total:,}")

# 3. pct_of_channel_vintage_accounts sums to 1.0 per (vintage_month, channel).
w = Window.partitionBy("vintage_month", "channel")
bad_pct = (
    l2_tables["lfs_layer2_score_distribution"]
    .groupBy("vintage_month", "channel")
    .agg(F.sum("pct_of_channel_vintage_accounts").alias("total_pct"))
    .filter(F.abs(F.col("total_pct") - 1.0) > 1e-9)
    .count()
)
if bad_pct > 0:
    errors.append(f"[FAIL] {bad_pct} partition(s) where pct != 1.0")
else:
    print("[OK]   pct_of_channel_vintage_accounts sums to 1.0 in all partitions")

# 4. Both channels present in every Layer3 table.
for tname, tdf in l3_tables.items():
    if "channel" in tdf.columns:
        channels = {r["channel"] for r in tdf.select("channel").distinct().collect()}
        if channels != {"digital", "directmail"}:
            errors.append(f"[FAIL] {tname} missing channels: {channels}")

print("[OK]   Both channels present in all Layer3 tables")

# 5. PSI and KS are non-negative.
neg_psi = l3_tables["lfs_layer3_score_drift"].filter(F.col("score_psi") < 0).count()
neg_ks  = l3_tables["lfs_layer3_score_drift"].filter(F.col("score_ks") < 0).count()
if neg_psi or neg_ks:
    errors.append(f"[FAIL] Negative PSI={neg_psi} or KS={neg_ks}")
else:
    print("[OK]   PSI and KS are non-negative")

# 6. population_mix pct sums to 1.0 per (vintage_month, channel, segment_type).
bad_mix = (
    l3_tables["lfs_layer3_population_mix"]
    .groupBy("vintage_month", "channel", "segment_type")
    .agg(F.sum("pct_of_channel_accounts").alias("total_pct"))
    .filter(F.abs(F.col("total_pct") - 1.0) > 1e-9)
    .count()
)
if bad_mix > 0:
    errors.append(f"[FAIL] {bad_mix} population_mix partition(s) where pct != 1.0")
else:
    print("[OK]   population_mix pct sums to 1.0 in all partitions")

# 7. segment_value is always a string.
non_str = [
    r["segment_value"]
    for r in l3_tables["lfs_layer3_population_mix"].select("segment_value").collect()
    if not isinstance(r["segment_value"], str)
]
if non_str:
    errors.append(f"[FAIL] Non-string segment_value found: {non_str[:3]}")
else:
    print("[OK]   All segment_value entries are strings")

print()
if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print(" ", e)
else:
    print("All validation checks passed.")

Running validation checks...

[OK]   Layer1 row count matches source: 1,200
[OK]   Layer2 account_count sums to Layer1 total: 1,200
[OK]   pct_of_channel_vintage_accounts sums to 1.0 in all partitions
[OK]   Both channels present in all Layer3 tables
[OK]   PSI and KS are non-negative
[OK]   population_mix pct sums to 1.0 in all partitions
[OK]   All segment_value entries are strings

All validation checks passed.


## 9. Multi-Format Report

The `report_builder` module reads all Layer 2 and Layer 3 output DataFrames (including
performance and calibration tables when actuals are available) and generates a monitoring
report in **two formats**:

| Format | Path | Purpose |
|---|---|---|
| **Markdown** | `outputs/lfs_report_*.md` | Canonical source, version control |
| **HTML** | `outputs/lfs_report_*.html` | Charts, inline display, stakeholder distribution |

Report sections:
- **A.** Run metadata  
- **B.** Executive summary — model health, performance snapshot, top drivers
- **C.** Business summary
- **D.** Monitoring — D1 Score Drift, D2 Feature PSI, D3 Correlation Drift, D4 Population Mix, D5 Data Quality, **D6 Performance**, **D7 Calibration**
- **E.** Charts *(HTML only — requires matplotlib)*
- **F.** Flags and Diagnosis

> Install dependencies: `pip install markdown matplotlib`

In [28]:
from framework.report_builder import build_report, _detect_html_backend
import os

# Use the l3_tables that includes performance + calibration.
outputs = {config.layer1["output_table"]: enriched_df}
outputs.update(l2_tables)
outputs.update(l3_tables_with_perf)

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

report_paths = build_report(
    outputs=outputs,
    config=config,
    score_month=SCORE_MONTH,
    model_version=MODEL_VERSION,
    output_dir=OUTPUT_DIR,
)

print("Generated files:")
for fmt, path in report_paths.items():
    print(f"  {fmt.upper():<5} -> {path or 'skipped'}")

print()
print(f"HTML backend : {_detect_html_backend() or 'none'}")
print(f"Output dir   : {os.path.abspath(OUTPUT_DIR)}")

2026-03-18 14:48:54,776 [framework.report_builder] INFO: Building report for model=lfs, score_month=2025-05
INFO:framework.report_builder:Building report for model=lfs, score_month=2025-05
/Users/hq/dev/claudecode/model_reporting/framework/charts.py:79: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=rotation, ha="right")
/Users/hq/dev/claudecode/model_reporting/framework/charts.py:79: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=rotation, ha="right")
2026-03-18 14:48:56,360 [framework.charts] INFO: Charts built: 4
INFO:framework.charts:Charts built: 4
2026-03-18 14:48:56,362 [framework.report_builder] INFO: Markdown report -> /Users/hq/dev/claudecode/model_reporting/notebooks/outputs/lfs_report_2025-05_v1.0_2026-03-18.md
I

Generated files:
  MD    -> /Users/hq/dev/claudecode/model_reporting/notebooks/outputs/lfs_report_2025-05_v1.0_2026-03-18.md
  HTML  -> /Users/hq/dev/claudecode/model_reporting/notebooks/outputs/lfs_report_2025-05_v1.0_2026-03-18.html

HTML backend : markdown
Output dir   : /Users/hq/dev/claudecode/model_reporting/notebooks/outputs


In [29]:
# Display the HTML report inline in the notebook.
from IPython.display import display, HTML


if report_paths["md"]:
    # HTML not generated — render markdown file as fallback.
    from IPython.display import Markdown
    display(Markdown(open(report_paths["md"], encoding="utf-8").read()))
else:
    print("No report files were generated.")

# Loss Forecast Suite — Reporting Run 2025-05

> **Score month**: 2025-05 &nbsp;|&nbsp; **Model version**: v1.0 &nbsp;|&nbsp; **Generated**: 2026-03-18 21:48 UTC

---

## A. Run Metadata

| Property | Value |
|---|---|
| Model | lfs — Loss Forecast Suite |
| Score month | 2025-05 |
| Model version | v1.0 |
| Baseline window | 2024-08 → 2025-02 |
| Channels | digital, directmail |
| Features monitored | 11 |
| Generated | 2026-03-18 21:48 UTC |
| Layer 1 accounts (current) | 1,200 |

---

## B. Executive Summary

### Model Health

| Channel | Avg Score | PSI | KS | Health |
|---|---|---|---|---|
| digital | 0.4205 | 0.1201 [WARN] | 0.1274 [WARN] | **[WARN]** |
| directmail | 0.4035 | 0.4461 **[ALERT]** | 0.2603 [WARN] | **[ALERT]** |

**Overall model health: **[ALERT]****

### Performance Snapshot

| Channel | Predicted Bad Rate | Actual Bad Rate | Gap | EDR90 |
|---|---|---|---|---|
| digital | 42.05% | 49.00% | +0.0695 | 49.00% |
| directmail | 40.35% | 45.00% | +0.0465 | 45.00% |

### Top Drifting Features

| Rank | Feature | Max PSI | Baseline Mean | Current Mean | Shift |
|---|---|---|---|---|---|
| 1 | feature_03 | 0.8483 **[ALERT]** | 0.3023 | 0.4346 | +0.1323 |
| 2 | feature_01 | 0.8375 **[ALERT]** | 0.3461 | 0.4526 | +0.1065 |
| 3 | feature_10 | 0.7268 **[ALERT]** | 0.4938 | 0.5727 | +0.0789 |
| 4 | feature_04 | 0.2539 **[ALERT]** | 0.3961 | 0.4569 | +0.0608 |
| 5 | feature_02 | 0.1084 [WARN] | 0.4650 | 0.4750 | +0.0100 |

### Key Drivers & Observations

- Score distribution has shifted significantly (max PSI = 0.4461) vs the baseline window. Investigate upstream feature or population changes.
- digital: right-tail expansion — 23.0% of accounts exceed the baseline P90 threshold. Score distribution is shifting toward higher-risk predictions.
- directmail: right-tail expansion — 30.5% of accounts exceed the baseline P90 threshold. Score distribution is shifting toward higher-risk predictions.
- Feature drift concentrated in: feature_03 (PSI=0.8483), feature_01 (PSI=0.8375), feature_10 (PSI=0.7268). Review upstream data pipelines for these inputs.
- Calibration gap of 0.339 detected in at least one score bin. Predicted probabilities deviate from observed bad rates — review the model recalibration schedule.
- digital: model is underestimating risk at the channel level (predicted 42.0%, actual 49.0%, gap = +0.0695).
- directmail: model is underestimating risk at the channel level (predicted 40.4%, actual 45.0%, gap = +0.0465).

---

## C. Business Summary

### Overall Volume & Score (2025-05)

| Channel | Accounts | Avg Score | Avg Receivable | Avg Sale Amount |
|---|---|---|---|---|
| digital | 200 | 0.4205 | 7283.25 | 2844.92 |
| directmail | 200 | 0.4035 | 5841.38 | 2445.50 |

### Score Band Distribution (2025-05)

Decile bins 1–3 = Low · 4–7 = Medium · 8–10 = High

| Channel | Low (D1–D3) | Medium (D4–D7) | High (D8–D10) |
|---|---|---|---|
| digital | 30.0% | 40.0% | 30.0% |
| directmail | 30.0% | 40.0% | 30.0% |

### Score by Acquisition Source (2025-05)

| Channel | Source | Accounts | Avg Score |
|---|---|---|---|
| digital | organic | 83 | 0.4245 |
| digital | paid | 69 | 0.4123 |
| digital | referral | 48 | 0.4252 |
| directmail | organic | 74 | 0.3913 |
| directmail | paid | 67 | 0.3988 |
| directmail | referral | 59 | 0.4244 |

### Score by Product Line (2025-05)

| Channel | Line | Accounts | Avg Score |
|---|---|---|---|
| digital | business | 82 | 0.4395 |
| digital | personal | 118 | 0.4072 |
| directmail | business | 38 | 0.3759 |
| directmail | personal | 162 | 0.4100 |

---

## D. Monitoring Summary

### D1. Score Drift (2025-05)

| Channel | Mean | Std | PSI | KS | P50 | P90 | P95 | Top-10% Mean | % Above P90 Base | % Above P95 Base |
|---|---|---|---|---|---|---|---|---|---|---|
| digital | 0.4205 | 0.2115 | 0.1201 [WARN] | 0.1274 [WARN] | 0.3887 | 0.6949 | 0.7870 | 0.7949 | 23.0% [WARN] | 18.0% [WARN] |
| directmail | 0.4035 | 0.2081 | 0.4461 **[ALERT]** | 0.2603 [WARN] | 0.4038 | 0.6429 | 0.6977 | 0.7510 | 30.5% [WARN] | 22.5% [WARN] |

### D2. Feature PSI — Top 5 per Channel (2025-05)

| Channel | Feature | PSI | Base Mean | Curr Mean | Mean Shift | Base Std | Curr Std |
|---|---|---|---|---|---|---|---|
| digital | feature_03 | 0.7228 **[ALERT]** | 0.2983 | 0.4273 | +0.1291 | 0.1487 | 0.1511 |
| digital | feature_10 | 0.7117 **[ALERT]** | 0.4965 | 0.5595 | +0.0630 | 0.2329 | 0.2242 |
| digital | feature_01 | 0.4405 **[ALERT]** | 0.3804 | 0.4592 | +0.0789 | 0.1115 | 0.1343 |
| digital | feature_04 | 0.2027 [WARN] | 0.4195 | 0.4746 | +0.0550 | 0.1250 | 0.1423 |
| digital | feature_02 | 0.1084 [WARN] | 0.4650 | 0.4750 | +0.0100 | 0.1503 | 0.1680 |
| directmail | feature_03 | 0.8483 **[ALERT]** | 0.3023 | 0.4346 | +0.1323 | 0.1446 | 0.1518 |
| directmail | feature_01 | 0.8375 **[ALERT]** | 0.3461 | 0.4526 | +0.1065 | 0.1025 | 0.1308 |
| directmail | feature_10 | 0.7268 **[ALERT]** | 0.4938 | 0.5727 | +0.0789 | 0.2331 | 0.2238 |
| directmail | feature_04 | 0.2539 **[ALERT]** | 0.3961 | 0.4569 | +0.0608 | 0.1239 | 0.1312 |
| directmail | feature_11 | 0.0705 | 0.4986 | 0.4720 | -0.0266 | 0.2325 | 0.2256 |

### D3. Feature–Score Correlation Drift — Top 5 (2025-05)

| Channel | Feature | Baseline Corr | Current Corr | Change |
|---|---|---|---|---|
| digital | feature_04 | 0.4939 | 0.6841 | +0.1902 |
| digital | feature_11 | 0.0250 | -0.1362 | -0.1612 |
| directmail | feature_02 | 0.2998 | 0.4544 | +0.1546 |
| directmail | feature_04 | 0.4790 | 0.6005 | +0.1215 |
| directmail | feature_01 | 0.7900 | 0.9089 | +0.1189 |

### D4. Population Mix — Source Segment (2025-05)

| Channel | Source | Accounts | % of Channel |
|---|---|---|---|
| digital | organic | 83 | 41.5% |
| digital | paid | 69 | 34.5% |
| digital | referral | 48 | 24.0% |
| directmail | organic | 74 | 37.0% |
| directmail | paid | 67 | 33.5% |
| directmail | referral | 59 | 29.5% |

### D5. Data Quality (2025-05)

| Channel | Missing Score Rate | Outlier Score Rate |
|---|---|---|
| digital | 0.000% | 0.000% |
| directmail | 0.000% | 0.000% |

### D6. Performance Monitoring (2025-05)

| Channel | Accounts | Avg Score | Predicted Bad Rate | Actual Bad Rate | Gap | EDR30 | EDR60 | EDR90 |
|---|---|---|---|---|---|---|---|---|
| digital | 200 | 0.4205 | 42.05% | 49.00% | +0.0695 | 59.00% | 50.00% | 49.00% |
| directmail | 200 | 0.4035 | 40.35% | 45.00% | +0.0465 | 57.50% | 51.00% | 45.00% |

### D7. Calibration (2025-05)

| Channel | Score Decile | Accounts | Predicted Rate | Actual Rate | Gap |
|---|---|---|---|---|---|
| digital | 1 | 19 | 7.58% | 10.53% | +0.0295 |
| digital | 2 | 18 | 20.10% | 22.22% | +0.0212 |
| digital | 3 | 19 | 25.79% | 31.58% | +0.0579 |
| digital | 4 | 19 | 31.49% | 31.58% | +0.0009 |
| digital | 5 | 19 | 35.66% | 26.32% | -0.0934 |
| digital | 6 | 14 | 40.08% | 57.14% | +0.1706 |
| digital | 7 | 16 | 43.39% | 50.00% | +0.0661 |
| digital | 8 | 14 | 49.06% | 78.57% | +0.2951 |
| digital | 9 | 16 | 55.23% | 56.25% | +0.0102 |
| digital | 10 | 46 | 72.00% | 84.78% | +0.1278 |
| directmail | 1 | 23 | 4.54% | 4.35% | -0.0019 |
| directmail | 2 | 7 | 15.92% | 42.86% | +0.2694 |
| directmail | 3 | 12 | 20.50% | 16.67% | -0.0383 |
| directmail | 4 | 7 | 25.56% | 28.57% | +0.0301 |
| directmail | 5 | 17 | 29.84% | 41.18% | +0.1133 |
| directmail | 6 | 10 | 33.00% | 20.00% | -0.1300 |
| directmail | 7 | 10 | 36.38% | 20.00% | -0.1638 |
| directmail | 8 | 26 | 40.94% | 61.54% | +0.2060 |
| directmail | 9 | 27 | 46.99% | 48.15% | +0.0116 |
| directmail | 10 | 61 | 63.87% | 68.85% | +0.0498 |

---

## E. Charts

*Charts are embedded in the HTML version of this report.*
*Install matplotlib and re-generate to include visual output: `pip install matplotlib`*

---

## F. Flags and Diagnosis

### Flags and Observations

- [WARN]  digital: score PSI = 0.1201 — exceeds warning threshold 0.1
- [WARN]  digital: score KS = 0.1274 — exceeds threshold 0.1
- [WARN]  digital: 23.0% of accounts above baseline P90 (threshold 15.0%)
- [WARN]  digital: 18.0% of accounts above baseline P95 (threshold 15.0%)
- [ALERT] directmail: score PSI = 0.4461 — exceeds alert threshold 0.25
- [WARN]  directmail: score KS = 0.2603 — exceeds threshold 0.1
- [WARN]  directmail: 30.5% of accounts above baseline P90 (threshold 15.0%)
- [WARN]  directmail: 22.5% of accounts above baseline P95 (threshold 15.0%)
- [ALERT] Feature feature_01 (digital): PSI = 0.4405
- [WARN]  Feature feature_02 (digital): PSI = 0.1084
- [ALERT] Feature feature_03 (digital): PSI = 0.7228
- [WARN]  Feature feature_04 (digital): PSI = 0.2027
- [ALERT] Feature feature_10 (digital): PSI = 0.7117
- [INFO]  Null segment values replaced with 'Unknown' in: receivable_bucket, saleamount_bucket

### Automated Diagnosis

- Score distribution has shifted significantly (max PSI = 0.4461) vs the baseline window. Investigate upstream feature or population changes.
- digital: right-tail expansion — 23.0% of accounts exceed the baseline P90 threshold. Score distribution is shifting toward higher-risk predictions.
- directmail: right-tail expansion — 30.5% of accounts exceed the baseline P90 threshold. Score distribution is shifting toward higher-risk predictions.
- Feature drift concentrated in: feature_03 (PSI=0.8483), feature_01 (PSI=0.8375), feature_10 (PSI=0.7268). Review upstream data pipelines for these inputs.
- Calibration gap of 0.339 detected in at least one score bin. Predicted probabilities deviate from observed bad rates — review the model recalibration schedule.
- digital: model is underestimating risk at the channel level (predicted 42.0%, actual 49.0%, gap = +0.0695).
- directmail: model is underestimating risk at the channel level (predicted 40.4%, actual 45.0%, gap = +0.0465).

---

*Report generated by the model_reporting framework · lfs · v1.0*

## 10. V2 Production Monitoring Framework

The v2 framework introduces:
- **Running-month alignment** — performance metrics use maturity-aligned cohorts (M3/M6/M9/M12)
- **Hierarchical thresholds** — default → model → scorecard override chain
- **Dual reports** — Business (stakeholder-friendly) + MMR (technical governance)
- **Score snapshot mart + Performance mart** — production data model
- **Scorecard support** — per-scorecard monitoring (config-gated)
- **Sample size controls** — guards against unstable metric computation

> The v1 and v2 pipelines are independent. V2 does not modify or depend on L1/L2/L3 outputs.

In [30]:
import importlib
import framework.v2.mock as _v2_mock
importlib.reload(_v2_mock)
from framework.v2.mock import build_mock_score_mart, build_mock_perf_mart

# Build v2 data marts from the FULL history (baseline + current).
# baseline_df spans 2024-08 to 2025-02  (from Section 2)
# current_df  spans 2025-03 to 2025-05  (from Section 2)
score_mart = build_mock_score_mart(spark, baseline_df, current_df, model_version=MODEL_VERSION)
perf_mart  = build_mock_perf_mart(spark, score_mart, seed=42)

print("\n=== Score Snapshot Mart ===")
print(f"  Rows: {score_mart.count():,}   Columns: {len(score_mart.columns)}")
score_mart.select("creditaccountid", "score_month", "score", "channel", "scorecard_id",
                  "score_decile_static", "is_excluded_from_monitoring").show(5, truncate=False)

print("\n=== Performance Mart ===")
print(f"  Rows: {perf_mart.count():,}   Columns: {len(perf_mart.columns)}")
perf_mart.select("creditaccountid", "score_month",
                 "edr30_m3", "edr60_m6", "edr90_m9", "co_m12",
                 "is_mature_m3", "is_mature_m9").show(5, truncate=False)


2026-03-18 14:48:56,622 [framework.v2.mock] INFO: Mock score mart: 4700 rows, 10 months, 94 excluded
INFO:framework.v2.mock:Mock score mart: 4700 rows, 10 months, 94 excluded


Score mart: 4,700 rows across 10 months
  Months: ['2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05']
  Excluded from monitoring: 94 (2.0%)


2026-03-18 14:48:56,900 [framework.v2.mock] INFO: Mock perf mart: 4700 rows, 10 months
INFO:framework.v2.mock:Mock perf mart: 4700 rows, 10 months


Perf mart: 4,700 rows across 10 months  (M9 bad rate: 36.4%)

=== Score Snapshot Mart ===
  Rows: 4,700   Columns: 30
+---------------+-----------+-----+-------+------------+-------------------+---------------------------+
|creditaccountid|score_month|score|channel|scorecard_id|score_decile_static|is_excluded_from_monitoring|
+---------------+-----------+-----+-------+------------+-------------------+---------------------------+
|15             |2024-08    |0.01 |digital|SC001       |1                  |0                          |
|216            |2024-08    |0.01 |digital|SC001       |1                  |0                          |
|244            |2024-08    |0.01 |digital|SC001       |1                  |0                          |
|512            |2024-09    |0.01 |digital|SC001       |1                  |0                          |
|716            |2024-09    |0.01 |digital|SC001       |1                  |0                          |
+---------------+-----------+-----+-------

In [31]:
# ── Pre-run validation: confirm the v2 data is complete ──────────
from framework.v2.cohort import get_maturity_mapping

REPORTING_MONTH = "2025-05"

# A. Score months present
score_months = sorted([r['score_month'] for r in score_mart.select('score_month').distinct().collect()])
print(f"A. Score months in score_mart ({len(score_months)}):")
print(f"   {score_months}")

# B. Baseline window
baseline_rows = score_mart.filter(
    (F.col('score_month') >= '2024-08') & (F.col('score_month') <= '2025-02')
).count()
print(f"\nB. Baseline window [2024-08, 2025-02]: {baseline_rows:,} rows")
assert baseline_rows > 0, "FAIL: baseline window is empty!"

# C. Current cohort
current_rows = score_mart.filter(F.col('score_month') == REPORTING_MONTH).count()
print(f"\nC. Current cohort (score_month={REPORTING_MONTH}): {current_rows:,} rows")
assert current_rows > 0, "FAIL: current cohort is empty!"

# D. Maturity cohorts
mat_map = get_maturity_mapping(REPORTING_MONTH)
print(f"\nD. Maturity cohort alignment:")
for label, sm in mat_map.items():
    n_score = score_mart.filter(F.col('score_month') == sm).count()
    n_perf  = perf_mart.filter(F.col('score_month') == sm).count()
    n_join  = (
        score_mart.filter(F.col('score_month') == sm)
        .join(perf_mart.filter(F.col('score_month') == sm),
              on=['creditaccountid', 'score_month'], how='inner')
        .count()
    )
    status = 'OK' if n_join > 0 else ('no data' if n_score == 0 else 'join=0')
    print(f"   {label}: score_month={sm}  score={n_score:>5}  perf={n_perf:>5}  joined={n_join:>5}  [{status}]")

print("\n=== All pre-run checks passed ===")


A. Score months in score_mart (10):
   ['2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05']

B. Baseline window [2024-08, 2025-02]: 3,500 rows


2026-03-18 14:48:57,351 [framework.v2.cohort] INFO: Maturity mapping for 2025-05: {'M3': '2025-02', 'M6': '2024-11', 'M9': '2024-08', 'M12': '2024-05'}
INFO:framework.v2.cohort:Maturity mapping for 2025-05: {'M3': '2025-02', 'M6': '2024-11', 'M9': '2024-08', 'M12': '2024-05'}



C. Current cohort (score_month=2025-05): 400 rows

D. Maturity cohort alignment:
   M3: score_month=2025-02  score=  500  perf=  500  joined=  500  [OK]
   M6: score_month=2024-11  score=  500  perf=  500  joined=  500  [OK]
   M9: score_month=2024-08  score=  500  perf=  500  joined=  500  [OK]
   M12: score_month=2024-05  score=    0  perf=    0  joined=    0  [no data]

=== All pre-run checks passed ===


In [32]:
from framework.v2.runner import run_monitoring

# REPORTING_MONTH defined in validation cell above.
v2_result = run_monitoring(
    score_mart=score_mart,
    perf_mart=perf_mart,
    reporting_month=REPORTING_MONTH,
    config="/Users/hq/dev/claudecode/model_reporting/conf/models/lfs_v2.yaml",
    output_dir="outputs",
)

print(f"Model:            {v2_result.model_name}")
print(f"Reporting month:  {v2_result.reporting_month}")
print(f"Run timestamp:    {v2_result.run_timestamp}")
print(f"Baseline:         {v2_result.baseline_info.get('type')} "
      f"[{v2_result.baseline_info.get('start_month')} → {v2_result.baseline_info.get('end_month')}] "
      f"({v2_result.baseline_info.get('account_count', 0):,} accounts)")
print(f"Governance flags: {len(v2_result.flags)}")
print()

# Cohort summary
print("=== Cohort Summary ===")
for label, info in v2_result.cohort_info.items():
    status = "available" if info["available"] else info.get("note", "not available")
    print(f"  {label}: score_month={info['score_month']}  "
          f"scored={info['account_count']}  mature={info['mature_count']}  ({status})")


2026-03-18 14:48:58,298 [framework.v2.runner] INFO: Computing baseline for 2025-05
INFO:framework.v2.runner:Computing baseline for 2025-05
2026-03-18 14:48:58,435 [framework.v2.data_model] INFO: Monitoring filter: excluded 94 of 4700 accounts
INFO:framework.v2.data_model:Monitoring filter: excluded 94 of 4700 accounts
2026-03-18 14:48:58,599 [framework.v2.baseline] INFO: Fixed-window baseline [2024-08, 2025-02]: 3429 accounts across 7 months
INFO:framework.v2.baseline:Fixed-window baseline [2024-08, 2025-02]: 3429 accounts across 7 months
2026-03-18 14:48:58,599 [framework.v2.runner] INFO: Building current cohort for 2025-05
INFO:framework.v2.runner:Building current cohort for 2025-05
2026-03-18 14:48:58,712 [framework.v2.data_model] INFO: Monitoring filter: excluded 94 of 4700 accounts
INFO:framework.v2.data_model:Monitoring filter: excluded 94 of 4700 accounts
2026-03-18 14:48:58,776 [framework.v2.cohort] INFO: Current cohort (2025-05): 391 accounts
INFO:framework.v2.cohort:Current c

Model:            lfs
Reporting month:  2025-05
Run timestamp:    2026-03-18T14:48:58.298222
Baseline:         fixed_window [2024-08 → 2025-02] (3,429 accounts)
Governance flags: 84

=== Cohort Summary ===
  M3: score_month=2025-02  scored=485  mature=485  (available)
  M6: score_month=2024-11  scored=497  mature=497  (available)
  M9: score_month=2024-08  scored=491  mature=491  (available)


In [33]:
print("=== Stability Metrics (overall) ===\n")
stab = v2_result.stability.get("overall", {})
print(f"Score PSI: {stab.get('score_psi', 0):.4f}  [{stab.get('score_psi_status', 'N/A')}]")
print()

print("Per-channel:")
for ch, ch_data in stab.get("channels", {}).items():
    print(f"  {ch:<12}  PSI={ch_data['score_psi']:.4f} [{ch_data['score_psi_status']}]  "
          f"mean={ch_data['mean_score']:.4f}  n={ch_data['account_count']}")
print()

print("Top drifting features:")
for feat in stab.get("feature_psi", [])[:5]:
    print(f"  {feat['feature_name']:<14}  PSI={feat['psi']:.4f} [{feat['status']}]  "
          f"shift={feat.get('mean_shift', 0):.4f}" if feat.get('mean_shift') is not None else
          f"  {feat['feature_name']:<14}  PSI={feat['psi']:.4f} [{feat['status']}]")

=== Stability Metrics (overall) ===

Score PSI: 0.1747  [WARNING]

Per-channel:
  digital       PSI=0.1269 [WARNING]  mean=0.4227  n=196
  directmail    PSI=0.4519 [ALERT]  mean=0.4080  n=195

Top drifting features:
  feature_03      PSI=0.7350 [ALERT]  shift=0.1297
  feature_10      PSI=0.7222 [ALERT]  shift=0.0699
  feature_01      PSI=0.5437 [ALERT]  shift=0.0945
  feature_04      PSI=0.2077 [WARNING]  shift=0.0592
  feature_11      PSI=0.0404 [OK]  shift=-0.0141


In [34]:
print("=== Performance Metrics ===\n")
for sc_id, rows in v2_result.performance.items():
    if sc_id != "overall":
        continue
    for row in rows:
        if row.get("note"):
            print(f"  {row.get('maturity','?'):>4}  {row.get('channel','?'):<12}  {row['note']}")
            continue
        edr = row.get("edr")
        bad = row.get("bad_rate")
        print(f"  {row.get('maturity','?'):>4}  {row.get('channel','?'):<12}  "
              f"n={row.get('account_count',0):>5}  "
              f"EDR={edr:.2%}" + (f"  bad_rate={bad:.2%}" if bad is not None else ""))

print("\n=== Calibration (M9, overall) ===\n")
calib = v2_result.calibration.get("overall", {}).get("M9", [])
if calib:
    all_ch = [r for r in calib if r.get("channel") == "all"]
    print(f"{'Bin':>5}  {'Accounts':>8}  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}  Status")
    print("-" * 55)
    for r in sorted(all_ch, key=lambda x: x.get("score_bin", 0)):
        print(f"{r['score_bin']:>5}  {r['account_count']:>8}  "
              f"{r['predicted_rate']:>10.4f}  {r['actual_rate']:>8.4f}  "
              f"{r['calibration_gap']:>+8.4f}  {r.get('gap_status','')}")
else:
    print("  Calibration not available for M9")

=== Performance Metrics ===

    M3  all           n=  485  EDR=50.93%  bad_rate=50.93%
    M3  digital       n=  239  EDR=56.07%  bad_rate=56.07%
    M3  directmail    n=  246  EDR=45.93%  bad_rate=45.93%
    M6  all           n=  497  EDR=47.28%  bad_rate=47.28%
    M6  digital       n=  247  EDR=51.82%  bad_rate=51.82%
    M6  directmail    n=  250  EDR=42.80%  bad_rate=42.80%
    M9  all           n=  491  EDR=32.38%  bad_rate=32.38%
    M9  digital       n=  246  EDR=35.77%  bad_rate=35.77%
    M9  directmail    n=  245  EDR=28.98%  bad_rate=28.98%

=== Calibration (M9, overall) ===

  Bin  Accounts   Predicted    Actual       Gap  Status
-------------------------------------------------------
    1        48      0.0789    0.0833   +0.0044  OK
    2        53      0.1829    0.1509   -0.0319  WARNING
    3        51      0.2497    0.1569   -0.0928  ALERT
    4        46      0.2949    0.3478   +0.0529  ALERT
    5        57      0.3433    0.2456   -0.0977  ALERT
    6        59   

In [35]:
print(f"=== Governance Flags ({len(v2_result.flags)} total) ===\n")
if v2_result.flags:
    print(f"{'Scorecard':<10}  {'Metric':<30}  {'Value':>8}  {'Status':<8}  Source")
    print("-" * 75)
    for f in sorted(v2_result.flags, key=lambda x: (x["status"], x["metric"])):
        val = f"{f['value']:.4f}" if isinstance(f['value'], float) else str(f['value'])
        print(f"{f['scorecard']:<10}  {f['metric']:<30}  {val:>8}  {f['status']:<8}  {f['source']}")
else:
    print("  No flags — all metrics within thresholds.")

=== Governance Flags (84 total) ===

Scorecard   Metric                             Value  Status    Source
---------------------------------------------------------------------------
overall     calibration_gap:M3:bin10          0.2379  ALERT     calibration
overall     calibration_gap:M3:bin10          0.2201  ALERT     calibration
overall     calibration_gap:M3:bin10          0.2756  ALERT     calibration
overall     calibration_gap:M3:bin2           0.1310  ALERT     calibration
overall     calibration_gap:M3:bin2           0.1761  ALERT     calibration
overall     calibration_gap:M3:bin2           0.1010  ALERT     calibration
overall     calibration_gap:M3:bin3           0.0656  ALERT     calibration
overall     calibration_gap:M3:bin4           0.0806  ALERT     calibration
overall     calibration_gap:M3:bin4           0.1418  ALERT     calibration
overall     calibration_gap:M3:bin5           0.2360  ALERT     calibration
overall     calibration_gap:M3:bin5           0.2056  AL

In [36]:
print("=== Generated Reports ===\n")
print(f"  Business: {v2_result.business_report or 'not generated'}")
print(f"  MMR:      {v2_result.mmr_report or 'not generated'}")

# # Display the MMR report inline
# from IPython.display import display, HTML

# mmr_path = v2_result.mmr_report
# if mmr_path:
#     print(f"\nDisplaying MMR report inline...\n")
#     display(HTML(open(mmr_path, encoding="utf-8").read()))
# elif v2_result.business_report:
#     print(f"\nDisplaying Business report inline...\n")
#     display(HTML(open(v2_result.business_report, encoding="utf-8").read()))
# else:
#     print("\nNo HTML reports generated. Install markdown: pip install markdown")

=== Generated Reports ===

  Business: /Users/hq/dev/claudecode/model_reporting/notebooks/outputs/lfs_business_2025-05_20260318.html
  MMR:      /Users/hq/dev/claudecode/model_reporting/notebooks/outputs/lfs_mmr_2025-05_20260318.html


In [37]:
print("score count:", score_mart.count())
print("perf count:", perf_mart.count())

# 看 join 后
joined = score_mart.join(perf_mart, "creditaccountid", "left")
print("joined count:", joined.count())

# 看 maturity
joined.selectExpr(
    "sum(case when is_mature_m3 = 1 then 1 else 0 end) as m3",
    "sum(case when is_mature_m6 = 1 then 1 else 0 end) as m6",
    "sum(case when is_mature_m9 = 1 then 1 else 0 end) as m9"
).show()

score count: 4700
perf count: 4700
joined count: 4700
+----+----+----+
|  m3|  m6|  m9|
+----+----+----+
|4700|4700|4700|
+----+----+----+



In [38]:
score_mart.select("score_month").distinct().orderBy("score_month").show(50, truncate=False)

+-----------+
|score_month|
+-----------+
|2024-08    |
|2024-09    |
|2024-10    |
|2024-11    |
|2024-12    |
|2025-01    |
|2025-02    |
|2025-03    |
|2025-04    |
|2025-05    |
+-----------+



## 8. Next Steps

### Connecting to real data

Replace the mock-data cells with production source reads.
The simplest notebook workflow once your environment is configured:

```python
from framework.runner import run

outputs = run(
    model="lfs",
    score_month="2025-05",
    model_version="v1.0",
    spark=spark,
    write_output=False,    # dry-run first
    return_outputs=True,
)

# Inspect outputs here, then re-run with write_output=True to promote.
```

### Things to verify on first real run

1. Row counts match the source table exactly.
2. Both channels have accounts in every vintage.
3. `dq_missing_flag` rate is below your SLA threshold.
4. `score_psi` is near zero for months within the baseline window.
5. `feature_03` (or whichever features are known to drift) shows expected PSI.

### Adding a second model

Drop a new YAML file in `conf/models/` and call `run(model="new_model", ...)`.
The framework requires no code changes.

See `docs/validation_checklist.md` for the full SQL-based QA checklist.